# 03 - Park effects by batted-ball type

**Method:** on balls in play, `woba_gap = actual wOBA - expected wOBA`. Expected wOBA comes from exit velocity and launch angle and does not know the park, so the gap picks up park effects (plus defense, weather, and luck).

Park effects are measured for each **batter side x batted-ball type** (ground ball, line drive, fly ball, popup) and compared with the league average for the same kind of contact. Pitch archetypes and pitchers are then scored by the batted balls they produce.

*Why not measure park effects per pitch archetype directly? We tried: the archetype-specific part of those effects had a split-half correlation of about 0.05 (noise). Measured by batted-ball type it is about 0.5. The stability check below reproduces both.*

In [ ]:
import sys
sys.path.append("..")

import pandas as pd
import matplotlib.pyplot as plt
from src.config import PROCESSED, INTERIM, FIGURES
from src.park_effects import contact_frame, park_bb_effects, split_half_stability
from src.fit import archetype_park_edges, best_archetypes_for_park, archetype_batted_ball_profile
from src.plots import plot_park_bb, plot_park_archetypes

df = pd.read_parquet(INTERIM / "pitches_archetyped.parquet")
contact = contact_frame(df)
print(f"{len(contact):,} balls in play")

## Step 1: park effects by batter side and batted-ball type

`pitcher_edge` is in wOBA per ball in play (multiply by 1000 for points). Positive means the park helps pitchers on that kind of contact.

In [ ]:
effects = park_bb_effects(contact, shrink_k=200)
effects.to_csv(PROCESSED / "park_bb_effects.csv", index=False)
print(f"{len(effects)} park/side/type cells; fewest balls in play in a cell: {effects['bip'].min()}")
effects.head()

## Sanity check: park-level ranking

Average gap by park, in wOBA points. Compare this ranking to the park factors leaderboard on Baseball Savant. Sutter Health Park has only 2 seasons of data and George M. Steinbrenner Field only 1, so treat those as low-confidence.

In [ ]:
(contact.groupby("park")["woba_gap"].mean() * 1000).sort_values(ascending=False).round(1)

## Stability check: do the effects repeat?

Effects are computed separately on the earlier seasons and on the later seasons, then correlated. "Specific part" removes each park's overall effect, leaving only how the park treats one kind of contact differently from another. That is the part the project's claim depends on.

In [ ]:
for level in ["archetype", "batted_ball"]:
    r = split_half_stability(contact, level=level)
    print(f"{level:12s} {r['cells']:4d} cells | overall corr {r['overall']:.2f} | specific-part corr {r['specific']:.2f}")

## Example park: Coors Field

In [ ]:
coors = effects[effects["park"] == "Coors Field"].sort_values(["stand", "bb_type"])
coors[["stand", "bb_type", "bip", "pitcher_edge", "hr_diff"]].round(3)

In [ ]:
plot_park_bb(effects, "Coors Field", path=FIGURES / "Coors_Field_batted_balls.png")
plt.show()

## Step 2: which pitch archetypes suit each park?

Each archetype (and pitcher hand) has a mix of batted balls by batter side and type. Weighting a park's effects by that mix gives the archetype's expected park effect.

- `edge_pts`: wOBA points on contact vs. a league-average park (positive helps the pitcher)
- `fit_pts`: `edge_pts` with the park's overall effect and the archetype's overall tendency removed, so it measures how much better this park suits the pitch than it suits a typical pitch (positive = better fit)

In [ ]:
arch = archetype_park_edges(contact, effects, min_bip=2000)
arch.to_csv(PROCESSED / "archetype_park_fit.csv", index=False)
archetype_batted_ball_profile(contact).to_csv(PROCESSED / "archetype_batted_ball_profile.csv", index=False)
print(arch["coverage"].min().round(3), "= lowest coverage (should be close to 1)")
best_archetypes_for_park(arch, "Coors Field", hand="R")

In [ ]:
plot_park_archetypes(arch, "Coors Field", "R", path=FIGURES / "Coors_Field_RHP.png")
plt.show()

## Make charts for every park

In [ ]:
for park in sorted(effects["park"].unique()):
    stem = park.replace(" ", "_")
    figs = [plot_park_bb(effects, park, path=FIGURES / f"{stem}_batted_balls.png")]
    for hand in ["R", "L"]:
        figs.append(plot_park_archetypes(arch, park, hand, path=FIGURES / f"{stem}_{hand}HP.png"))
    for fig in figs:
        if fig is not None:
            plt.close(fig)

## Commit

```
git add src notebooks data/processed slides/figures
git commit -m "park effects by batted-ball type, archetype fit scores"
git push
```